# Myllia - echoes of silenced genes
---

**authors**: [fsb2210](https://www.kaggle.com/fsb2210), [julianc93](https://www.kaggle.com/julianc93)

The task is to train a model that is able to predict *expression changes in scRNA-seq data induced by CRISPRi perturbations*. For that, we have a dataset of 80 different perturbations and the *average expression values* of genes, plus an unperturbed case (*non-targeting sgRNA*).

## Introduction

This problem esentially consists of inputs of strings given by the `pert_symbol` column and an output vector space of dimension equal to the number of columns minus the one corresponding to the `pert_symbol` (i.e., a 5127-dimensional space).

Clearly, this dataset needs to be preprocessed. In particular, our preprocessing setup will be as follows:

- treat each of the genes (perturbation and output ones) as nodes of a large network. Bonds between nodes will represent (undirected) protein-protein interactions (PPIs) and (directed) transcription factor (TF) - target interactions,
- compute delta averages for each of the output vectors ($\Delta_{\rm avg}$),
- train a machine learning (ML) model using the preprocessed datasets created during steps 1 and 2,
- predict $\Delta_{\rm avg}$ on new genes using the validation set (`pert_ids_val.csv`).

> **NOTE**
> 
> Dependencies list is: matplotlib numpy pandas tqdm scipy scikit-learn seaborn

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
import seaborn as sns
from tqdm import tqdm
import time
import warnings
warnings.filterwarnings("ignore")

from sklearn.impute import KNNImputer

Define global options:

In [2]:
experiment_name = ""

# directory with data files
data_dir = "../data"
working_dir = "../data"

# random state integer value for reproducibility concerns
random_state = 42

# threshold value to consider PPI interactions ONLY if score is larger than this
score_threshold = 0.75

# restart probability of continuing the perturbation in the network. if this is an array, compute every one of them and concatenate results
alpha_ppi = 0.85

# strength of regulatory factors
alpha_reg = 0.85

## Data loading

In [3]:
# links between proteins
links_df = pd.read_csv(f"{working_dir}/links.csv")
# protein name -> stringID
info_df = pd.read_csv(f"{working_dir}/info.csv")
# transcription factor - target interactions
tf_df = pd.read_csv(f"{working_dir}/tf_interactions.csv")

In [4]:
# train & validation sets
raw_train_df = pd.read_csv(f"{data_dir}/training_data_means.csv")
val_df = pd.read_csv(f"{data_dir}/pert_ids_val.csv")

In [5]:
# K562 set
raw_k562_df = pd.read_csv(f"{data_dir}/features_K562_gwps.csv")

In [6]:
# Hepg2 set
raw_hepg2_df = pd.read_csv(f"{data_dir}/features_hepg2_gwps.csv")

### Preprocessing steps

Remove perturbations that are present in the external datasets but have already beed defined inside the training sample:

In [7]:
# get perturbation genes from training sample
train_pert_genes = raw_train_df.pert_symbol.tolist()[:-1]

# remove duplicated perturbations in K562 sample
rows_to_drop = []
for k in raw_k562_df.index:
    gene = raw_k562_df.pert_symbol.iloc[k]
    if gene in train_pert_genes:
        rows_to_drop.append(k)

raw_k562_df.drop(rows_to_drop, inplace=True)
raw_k562_df.reset_index(drop=True, inplace=True)

# remove duplicated perturbations in HepG2 sample
rows_to_drop = []
for k in raw_hepg2_df.index:
    gene = raw_hepg2_df.pert_symbol.iloc[k]
    if gene in train_pert_genes:
        rows_to_drop.append(k)

raw_hepg2_df.drop(rows_to_drop, inplace=True)
raw_hepg2_df.reset_index(drop=True, inplace=True)

Impute missing values in K562 and HepG2 datasets, using nearest beighbors in the training sample:

In [8]:
# values to impute + ground truth
Y_true = raw_train_df.iloc[:, 1:].values
k562_ext = raw_k562_df.iloc[:, 1:].values
hepg2_ext = raw_hepg2_df.iloc[:, 1:].values

In [9]:
start_time = time.time()

# impute K562 data with training sample
imputer = KNNImputer(n_neighbors=10, weights="uniform")

train_w_k562 = np.vstack([Y_true, k562_ext])
train_w_k562_imputed = imputer.fit_transform(train_w_k562)
Y_k562 = train_w_k562_imputed[Y_true.shape[0]:, :]

# repeat impute but with HepG2 data with training sample
imputer = KNNImputer(n_neighbors=10, weights="uniform")

train_w_hepg2 = np.vstack([Y_true, hepg2_ext])
train_w_hepg2_imputed = imputer.fit_transform(train_w_hepg2)
Y_hepg2 = train_w_hepg2_imputed[Y_true.shape[0]:, :]

training_time = time.time() - start_time
print(f"Data imputation completed in {training_time:.2f} seconds ({training_time/60:.2f} minutes)")

Data imputation completed in 140.32 seconds (2.34 minutes)


Grab the name of the output genes:

In [10]:
output_genes = raw_train_df.columns[1:]

Apply imputated values to K562 and HepG2 datasets:

In [11]:
raw_k562_df[output_genes] = Y_k562
raw_hepg2_df[output_genes] = Y_hepg2

Compute delta expressions with respect to the `non-targeting` row for every dataset:

In [12]:
# train dataset
train_base = raw_train_df.iloc[-1,1:].values
train_df = raw_train_df[output_genes].apply(lambda x: x - train_base, axis=1)
train_df.insert(0, "pert_symbol", raw_train_df["pert_symbol"].copy())
train_df.drop([train_df.shape[0]-1], inplace=True)

# K562 dataset
k562_base = raw_k562_df.iloc[-1,1:].values
k562_df = raw_k562_df[output_genes].apply(lambda x: x - k562_base, axis=1)
k562_df.insert(0, "pert_symbol", raw_k562_df["pert_symbol"].copy())
k562_df.drop([k562_df.shape[0]-1], inplace=True)

# HepG2 dataset
hepg2_base = raw_hepg2_df.iloc[-1,1:].values
hepg2_df = raw_hepg2_df[output_genes].apply(lambda x: x - hepg2_base, axis=1)
hepg2_df.insert(0, "pert_symbol", raw_hepg2_df["pert_symbol"].copy())
hepg2_df.drop([hepg2_df.shape[0]-1], inplace=True)

Final step: concatenate each dataset into a single one:

In [13]:
full_train_df = pd.concat([train_df, k562_df, hepg2_df], ignore_index=True)
full_train_df.shape

(4473, 5128)

Create a dictionary mapping string ids from proteins to names. E.g., "9606.ENSP00000000233" -> "ARF5"

In [14]:
string_id_to_name = dict(zip(info_df["string_protein_id"], info_df["preferred_name"]))

Get the list of genes that **must** be present in the network

In [15]:
core_genes = []
train_pert_genes = full_train_df["pert_symbol"].tolist()
val_pert_genes = val_df["pert"].tolist()
core_genes.extend(train_pert_genes)
core_genes.extend(val_pert_genes)
core_genes.extend(output_genes)
core_genes = list(set(core_genes))
print(f"- total number of genes (perturbated + targets) in the training + validation sets (a.k.a., core_genes): {len(core_genes)}")

- total number of genes (perturbated + targets) in the training + validation sets (a.k.a., core_genes): 6834


## Feature and target engineering

We do some preprocessing on the input data to make it useful.

First, we work on the preprocessing of the genes using the PPIs obtained from **STRING-db**:

In [16]:
# remove unnecesary columns & compute weights between [0, 1]
edges_df = links_df.copy()
edges_df["weight"] = edges_df["combined_score"] / 1000.0
edges_df = edges_df.drop(columns=["combined_score"])
edges_df["protein1"] = edges_df["protein1"].map(string_id_to_name)
edges_df["protein2"] = edges_df["protein2"].map(string_id_to_name)
edges_df = edges_df.dropna(subset=["protein1", "protein2"])  # remove unmapped edges

# apply threshold on scores
edges = edges_df[edges_df["weight"] >= score_threshold]

# find all interactions where at least one of the proteins is in the core_genes list
mask = edges.protein1.isin(core_genes) | edges.protein2.isin(core_genes)
relevant_edges = edges[mask]

# create sets of genes with and without interactions, connected_genes & missing_core
connected_genes = set(relevant_edges.protein1.unique()) | set(relevant_edges.protein2.unique())
missing_core = set(core_genes) - connected_genes

# we store genes in a set to later use as a mapping
final_gene_set = connected_genes.copy()
final_gene_set.update(missing_core)

Next task is to repeat the analysis but this time using the TF-target interactions:

In [17]:
def get_edge_sign(row):
    """Assign signed weight based on stimulation/inhibition"""
    if row["is_stimulation"]:
        return +1.0
    elif row["is_inhibition"]:
        return -1.0
    elif row["consensus_stimulation"]:
        return +0.7
    elif row["consensus_inhibition"]:
        return -0.7
    else:
        return 0.3

# remove complex proteins
tf_clean = tf_df[~tf_df["source"].str.startswith("COMPLEX")]
# weight column according to the stimulation (inhibition)
tf_clean["sign"] = tf_clean.apply(get_edge_sign, axis=1)

# find all interactions where at least one of the genes is in the core_genes list
mask = tf_clean.source_genesymbol.isin(core_genes) | tf_clean.target_genesymbol.isin(core_genes)
relevant_tf = tf_clean[mask]

# create sets of genes with and without interactions, connected_genes & missing_core
connected_tf_genes = set(relevant_tf.source_genesymbol.unique()) | set(relevant_tf.target_genesymbol.unique())
missing_tf_core = set(core_genes) - connected_tf_genes

final_gene_set.update(connected_tf_genes)
final_gene_set.update(missing_tf_core)

Now we can have a final map between genes and indices:

In [18]:
# mapping between gene names and indices
gene_to_idx = {gene: idx for idx, gene in enumerate(final_gene_set)}

# and we can define the number of nodes!
n_nodes = len(gene_to_idx)

In [19]:
print(f"number of nodes in the graph: {n_nodes}")

number of nodes in the graph: 17062


### Network graph of genes

We are now ready to create the graph using adjacency matrices ($A$), one for the PPIs and one for the TF-target interactions:

In [20]:
# - protein-protein interactions
src = relevant_edges["protein1"].map(gene_to_idx).values
dst = relevant_edges["protein2"].map(gene_to_idx).values
w = relevant_edges["weight"].values

# weighted adjancency matrix
rows = np.concatenate([src, dst])
cols = np.concatenate([dst, src])
data = np.concatenate([w, w])
A_ppi = sp.csr_matrix((data, (rows, cols)), shape=(n_nodes, n_nodes))
# add a small self-loop to prevent a division by zero when computing D^-1.
eye = sp.eye(n_nodes, format="csr")
A_ppi = A_ppi + eye

# degree matrix
deg_ppi = np.array(A_ppi.sum(axis=1)).flatten()
# inverse of degree matrix
deg_inv_ppi = np.zeros_like(deg_ppi)
mask = deg_ppi > 0
deg_inv_ppi[mask] = 1 / deg_ppi[mask]
D_inv_ppi = sp.diags(deg_inv_ppi)

In [21]:
# - TF-target interactions
src = relevant_tf["source_genesymbol"].map(gene_to_idx).values
dst = relevant_tf["target_genesymbol"].map(gene_to_idx).values
w = relevant_tf["sign"].values

# weighted adjancency matrix
rows = np.concatenate([src, dst])
cols = np.concatenate([dst, src])
data = np.concatenate([w, w])
A_reg = sp.csr_matrix((data, (rows, cols)), shape=(n_nodes, n_nodes))

### Diffusion operator

Now we are ready to create the operator which is needed to solve diffusion for diffferent perturbations.

The operator is defined as:

\begin{equation}
    x = (I + \epsilon L) x_0
\end{equation}

where $x$ is the solution we seek, that is, it is the perturbation vector integrated over time (steady-state). $x_0$ is the initial perturbation vector which is a one-hot vector localized at the gene that is perturbed and $L = A D^{-1}$ is the so-called *Laplacian matrix*. This is defined for a single "timestep".

Thus, the integrated solution is found to be:

\begin{equation}
    x = (I - \alpha  A D^{-1})^{-1} x_0
\end{equation}

where $\alpha$ is a control parameter for the restart of the perturbation to its initial location in the graph. This solutions is known as the *random-walk with restart*.

We can solve the linear system using the LU decomposition:

In [22]:
start_time = time.time()

M_ppi = (sp.eye(n_nodes) - alpha_ppi * A_ppi.dot(D_inv_ppi))
LU_ppi = sp.linalg.splu(M_ppi.tocsc())

training_time = time.time() - start_time
print(f"LU decomposition completed in {training_time:.2f} seconds ({training_time/60:.2f} minutes)")

LU decomposition completed in 65.07 seconds (1.08 minutes)


Create auxiliary functions:

1. to one-hot encode vectors (`one_hot`),
2. to solve the diffusion equation (`diffuse_propagation`).

> NOTE: the diffusion on the TF-target interaction is separated into two channels: one for the *positive* interaction in which the interaction is an stimulation and a *negative* for the oppositve. This is to avoid problems where one gene is stimulated and inhibited at the same time.

In [23]:
def one_hot(gene, gene_to_idx):
    x0 = np.zeros(len(gene_to_idx))
    x0[gene_to_idx[gene]] = 1.0
    return x0

def diffuse_propagation_by_channels(gene):
    # initial perturbation (one-hot encoded)
    x0 = one_hot(gene, gene_to_idx)
    # --- PPI diffusion
    x = LU_ppi.solve(x0)
    # x0 = one_hot(gene, tmp_map)
    # x = x0.copy()
    # --- regulatory propagation
    # 1. split A_reg into positive and negative parts
    A_pos = A_reg.copy()
    A_pos.data = np.where(A_pos.data > 0, A_pos.data, 0)
    A_pos.eliminate_zeros()
    A_neg = A_reg.copy()
    A_neg.data = np.where(A_neg.data < 0, np.abs(A_neg.data), 0)
    A_neg.eliminate_zeros()
    # 2. propagate separately
    activation_signal = A_pos.T.dot(x)
    inhibition_signal = A_neg.T.dot(x)
    combined_signal = A_reg.T.dot(x)
    # 3. concatenate all three as features
    # return np.concatenate([activation_signal, inhibition_signal, combined_signal])
    return activation_signal - inhibition_signal

### Training and validation features

Finally, we can create our input and output (`X`, `Y`) datasets to be used for training a machine learning model.

- `X` will be the result of a diffusive perturbation for each of the training perturbations,
- `Y` is the delta expressions of the 5127 genes of the training set.

Same idea for the `X_val` and `Y_val` for the validation set.

We include differentially expressed genes (DEGs) frequency and mean expression levels as features:

In [43]:
Y_train = train_df[output_genes].values
Y_k562 = k562_df[output_genes].values
Y_hepg2 = hepg2_df[output_genes].values

In [44]:
deg_freq_train = np.mean(np.abs(Y_train) > 0.5, axis=0)
mean_expression_train = raw_train_df.iloc[-1,1:].values.mean(axis=0)

deg_freq_k562 = np.mean(np.abs(Y_k562) > 0.5, axis=0)
mean_expression_k562 = raw_k562_df.iloc[-1,1:].values.mean(axis=0)

deg_freq_hepg2 = np.mean(np.abs(Y_hepg2) > 0.5, axis=0)
mean_expression_hepg2 = raw_hepg2_df.iloc[-1,1:].values.mean(axis=0)

In [26]:
# stack DEG frequency and mean expressions
deg_freq_tile_train = np.tile(deg_freq_train, (train_df.shape[0], 1))
deg_freq_tile_k562 = np.tile(deg_freq_k562, (k562_df.shape[0], 1))
deg_freq_tile_hepg2 = np.tile(deg_freq_hepg2, (hepg2_df.shape[0], 1))
deg_freq = np.vstack([deg_freq_tile_train, deg_freq_tile_k562, deg_freq_tile_hepg2])

mean_exp_tile_train = np.tile(mean_expression_train, (train_df.shape[0], 1))
mean_exp_tile_k562 = np.tile(mean_expression_k562, (k562_df.shape[0], 1))
mean_exp_tile_hepg2 = np.tile(mean_expression_hepg2, (hepg2_df.shape[0], 1))
mean_exp = np.vstack([mean_exp_tile_train, mean_exp_tile_k562, mean_exp_tile_hepg2])

In [27]:
# stack diffusions
X_diff = np.vstack([diffuse_propagation_by_channels(g) for g in train_pert_genes])

Training features are ready, save them into `X_enh`

In [103]:
indices = [gene_to_idx[gene] for gene in output_genes]

# if reduce_input_dim_by_variance:
gene_var = np.var(X_diff[:, indices], axis=0)
var_threshold = np.percentile(gene_var, 80)
selected_mask = gene_var >= var_threshold
top_ind = np.where(selected_mask)[0]
X_final = X_diff[:, top_ind]

X_enh = np.hstack([X_final, deg_freq, mean_exp])
X_enh.shape

(4473, 6154)

Now we repeat the same process for the validation set

In [105]:
X_diff_val = np.vstack([diffuse_propagation_by_channels(g) for g in val_pert_genes])
deg_freq_tile_val = np.tile(deg_freq_train, (val_df.shape[0], 1))
mean_exp_tile_val = np.tile(mean_expression_train, (val_df.shape[0], 1))

X_enh_val = np.hstack([X_diff_val[:, top_ind], deg_freq_tile_val, mean_exp_tile_val])
X_enh_val.shape

(60, 6154)

And save them into files:

In [106]:
np.save("../data/processed/X_train.npy", X_enh)
np.save("../data/processed/X_val.npy", X_enh_val)

As our last step of the feature engineering, we save the training values into another file:

In [107]:
Y_train =  np.vstack([Y_train, Y_k562, Y_hepg2])

In [108]:
np.save("../data/processed/Y_train.npy", Y_train)

In [33]:
"""
from numpy.linalg import svd

# Compute singular values
U, s, Vt = svd(deg_freq, full_matrices=False)

# How many singular values matter?
cumsum = np.cumsum(s**2) / np.sum(s**2)
effective_rank = np.searchsorted(cumsum, 0.95) + 1

print(f"Effective rank (95% variance): {effective_rank}")
"""

'\nfrom numpy.linalg import svd\n\n# Compute singular values\nU, s, Vt = svd(deg_freq, full_matrices=False)\n\n# How many singular values matter?\ncumsum = np.cumsum(s**2) / np.sum(s**2)\neffective_rank = np.searchsorted(cumsum, 0.95) + 1\n\nprint(f"Effective rank (95% variance): {effective_rank}")\n'